In [1]:
import os, json, time, math, glob, numpy as np, torch
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train_ids_bpe.npy", recursive=True)[0])
print(DATA_DIR, os.listdir(DATA_DIR))

train_ids = np.load(os.path.join(DATA_DIR, "train_ids_bpe.npy"), mmap_mode="r")
val_ids   = np.load(os.path.join(DATA_DIR, "val_ids_bpe.npy"),   mmap_mode="r")
VOCAB_SIZE = 65355                                  # 65,354 BPE tokens + 1 EOS
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"train {len(train_ids):,} | val {len(val_ids):,} | max id {int(val_ids.max())} | vocab {VOCAB_SIZE} | {device}")


/kaggle/input/datasets/ashwaryrathore/cs781-legal-tokens ['val_ids_bpe.npy', 'train_ids_bpe.npy', 'legal_bpe_merges.json']
train 200,000,000 | val 5,000,000 | max id 65354 | vocab 65355 | cuda


In [2]:
%%writefile attention.py
#this is the same attention that i built in the A3 question 1


import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask",mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
       
        B, T, C = x.shape
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q * self.attention_scale) @ k.transpose(-2, -1)
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out
        


Writing attention.py


In [3]:
%%writefile model_arch.py

"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int =1024
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   
    attention_scale: float = None      
    norm_first: bool = False            

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    # TODO: implement PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    #               PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    pos = torch.arange(max_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    div = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos / div)
    pe[:, 1::2] = torch.cos(pos / div)
    return pe
    


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first  

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # and post-norm when norm_first=False.
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x
        


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=cfg.d_model ** -0.5)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        # TODO: run x through every block, then ln_f, then self.head
        x = self.tok_emb(idx) * math.sqrt(self.cfg.d_model)
        
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T]
            
        for block in self.blocks:
            x = block(x)
            
        x = self.ln_f(x)
        return self.head(x * self.cfg.d_model ** -0.5)
        

Writing model_arch.py


In [4]:
def count(V, d, L, d_ff):
    per_layer= 4*d*d+2*d*d_ff+d_ff+d+4*d
    total=V*d + L*per_layer + 2*d
    print(f"d={d} L={L} d_ff={d_ff} | embedding {V*d/1e6:.1f}M | layers {L*per_layer/1e6:.1f}M | total {total/1e6:.1f}M")

count(65355, 256, 4, 1024)
count(65355, 1024, 10, 1024)
count(65355, 1000, 10,4000 )

d=256 L=4 d_ff=1024 | embedding 16.7M | layers 3.2M | total 19.9M
d=1024 L=10 d_ff=1024 | embedding 66.9M | layers 63.0M | total 129.9M
d=1000 L=10 d_ff=4000 | embedding 65.4M | layers 120.1M | total 185.4M


In [5]:
import sys
for m in ("model_arch", "attention"): sys.modules.pop(m, None)
from model_arch import LMConfig, DecoderLM

cfg = LMConfig(vocab_size=VOCAB_SIZE, d_model=1000, n_layers=10, n_heads=8, n_kv_heads=8,
               d_ff=4000, max_seq_len=1024, dropout=0.1, norm_first=True)
model = DecoderLM(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

185,447,000 parameters


In [6]:
import math, torch, torch.nn.functional as F

model.eval()                                   # turn off dropout so outputs are deterministic
B, T = 4, 256
x = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
y = torch.randint(0, VOCAB_SIZE, (B, T), device=device)

with torch.no_grad():
    logits = model(x)
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
print("logits shape:", tuple(logits.shape), "| expected:", (B, T, VOCAB_SIZE))
print(f"initial loss: {loss.item():.3f} | expected ≈ ln(V) = {math.log(VOCAB_SIZE):.3f}")


x2 = x.clone(); x2[:, -1] = (x2[:, -1] + 1) % VOCAB_SIZE
with torch.no_grad():
    logits2 = model(x2)
print("causal mask OK:", torch.allclose(logits[:, :-1], logits2[:, :-1], atol=1e-4))

print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.2f} GB")
model.train()

logits shape: (4, 256, 65355) | expected: (4, 256, 65355)
initial loss: 11.087 | expected ≈ ln(V) = 11.088
causal mask OK: True
GPU memory in use: 1.33 GB


DecoderLM(
  (tok_emb): Embedding(65355, 1000)
  (blocks): ModuleList(
    (0-9): 10 x Block(
      (ln1): LayerNorm((1000,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=1000, out_features=1000, bias=False)
        (k_proj): Linear(in_features=1000, out_features=1000, bias=False)
        (v_proj): Linear(in_features=1000, out_features=1000, bias=False)
        (out_proj): Linear(in_features=1000, out_features=1000, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((1000,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=1000, out_features=4000, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=4000, out_features=1000, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (ln_f): Laye

In [7]:
def get_batch(split,B,T=1024):
    data=train_ids if split=="train" else val_ids
    starts=np.random.randint(0,len(data)-T-1,size=B)
    x=np.stack([data[s:s+T] for s in starts])
    y=np.stack([data[s+1:s+T+1]for s in starts])
    x=x.astype(np.int64)
    y=y.astype(np.int64)
    
    x=torch.from_numpy(x).to(device)
    y=torch.from_numpy(y).to(device)
    return x,y

In [8]:
xb, yb = get_batch("train", 2)
print(xb.shape, xb.dtype, xb.device)                          # expect torch.Size([2, 1024]) torch.int64 cuda:0
print("shifted by 1:", bool((xb[:, 1:] == yb[:, :-1]).all()))  # expect True

torch.Size([2, 1024]) torch.int64 cuda:0
shifted by 1: True


In [9]:
from dataclasses import asdict      # used in cell 11 to save cfg inside the checkpoint


MICRO_B = 4       
ACCUM =122
                  


MAX_STEPS =400    
                  


PEAK_LR=6.0e-4 
MIN_LR=6.0e-5  
WARMUP=40


WEIGHT_DECAY =0.1
CLIP=1.0  


EVAL_EVERY   = 25  
EVAL_BATCHES = 10   
LOG_EVERY    = 5   


CKPT_PATH  = "/kaggle/working/ckpt.pt"
FINAL_PATH = "/kaggle/working/final.pt"
TIME_LIMIT = 11 * 3600               

T = cfg.max_seq_len  

assert None not in (ACCUM, PEAK_LR, MIN_LR, WARMUP, WEIGHT_DECAY, CLIP)
assert WARMUP < MAX_STEPS, "warmup must be shorter than training"
print(f"tokens per optimizer step: {MICRO_B * ACCUM * T:,}")

tokens per optimizer step: 499,712


In [10]:
decay_params = [p for _, p in model.named_parameters() if p.dim() >= 2]
no_decay_params = [p for _, p in model.named_parameters() if p.dim() < 2]

optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0}
], lr=PEAK_LR)

def get_lr(step):
    if step < WARMUP:
        return PEAK_LR * (step + 1) / WARMUP
    if step >= MAX_STEPS:
        return MIN_LR
    decay_ratio = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * decay_ratio))

scaler = torch.amp.GradScaler("cuda")

num_decay = sum(p.numel() for p in decay_params)
num_no_decay = sum(p.numel() for p in no_decay_params)
total_params = num_decay + num_no_decay
print(f"decay {num_decay:,} | no-decay {num_no_decay:,} | total {total_params:,} (want 185,447,000)")
print(f"lr: start {get_lr(0):.2e} | peak {get_lr(WARMUP):.2e} | end {get_lr(MAX_STEPS):.2e}")

decay 185,355,000 | no-decay 92,000 | total 185,447,000 (want 185,447,000)
lr: start 1.50e-05 | peak 6.00e-04 | end 6.00e-05


In [11]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            x, y = get_batch(split, MICRO_B, T)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

print(estimate_loss())

{'train': 11.087858200073242, 'val': 11.088357925415039}


In [12]:
# ---------- find and load the step-100 checkpoint ----------
ck_path = glob.glob("/kaggle/input/**/ckpt*.pt", recursive=True) + glob.glob("/kaggle/working/ckpt_step100.pt")
print("checkpoint found:", ck_path)
ck = torch.load(ck_path[0], map_location="cpu")

model.load_state_dict(ck["model"])
optimizer.load_state_dict(ck["optimizer"])
scaler.load_state_dict(ck["scaler"])
start_step = ck["step"] + 1
best_val   = ck["best_val"]
history    = ck["history"]
del ck; torch.cuda.empty_cache()
print(f"resuming at step {start_step} | best val {best_val:.3f} | GPU mem {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("loaded model eval:", estimate_loss())          # expect ≈ 8.5–8.6

# ---------- training ----------
tokens_per_step = MICRO_B * ACCUM * T
t0 = t_log = time.time()
model.train()

def save(step, tag, path=CKPT_PATH):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "cfg": asdict(cfg),
                "best_val": best_val, "history": history}, path)
    print(f"  saved checkpoint ({tag}) at step {step} -> {path}")

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    for _ in range(ACCUM):
        x, y = get_batch("train", MICRO_B, T)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)) / ACCUM
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # ---- nan guard ----
    if not math.isfinite(loss.item()):
        print(f"  NaN/inf loss at step {step}: stopping. Last good checkpoint is safe.")
        break

    if step % LOG_EVERY == 0:
        torch.cuda.synchronize()
        dt = time.time() - t_log; t_log = time.time()
        print(f"step {step:5d} | loss {loss.item()*ACCUM:.3f} | lr {lr:.2e} | gnorm {gnorm:.2f} | "
              f"{tokens_per_step*LOG_EVERY/dt:,.0f} tok/s | peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        l = estimate_loss()
        for k, v in (("step", step), ("train", l["train"]), ("val", l["val"]), ("lr", lr)):
            history[k].append(v)
        print(f"  EVAL step {step} | train {l['train']:.3f} | val {l['val']:.3f} | val ppl {math.exp(l['val']):.1f}")
        if l["val"] < best_val:
            best_val = l["val"]
            save(step, "best")
        t_log = time.time()

    if time.time() - t0 > TIME_LIMIT:
        print(f"  time limit reached at step {step}")
        break

print(f"done in {(time.time()-t0)/60:.1f} min | best val {best_val:.3f}")
save(step, "final", FINAL_PATH)
with open("/kaggle/working/history.json", "w") as f:
    json.dump(history, f)
print("history saved:", len(history["step"]), "eval points")

checkpoint found: ['/kaggle/input/datasets/ashwaryrathore/arch1-ckpt-step100/ckpt.pt']
resuming at step 101 | best val 8.579 | GPU mem 2.8 GB
loaded model eval: {'train': 8.613154411315918, 'val': 8.611228942871094}
step   105 | loss 8.462 | lr 5.58e-04 | gnorm 0.15 | 7,139 tok/s | peak mem 11.0 GB
step   110 | loss 8.301 | lr 5.51e-04 | gnorm 0.15 | 7,081 tok/s | peak mem 11.0 GB
step   115 | loss 8.231 | lr 5.44e-04 | gnorm 0.15 | 7,083 tok/s | peak mem 11.0 GB
step   120 | loss 8.023 | lr 5.37e-04 | gnorm 0.15 | 7,081 tok/s | peak mem 11.0 GB
step   125 | loss 8.175 | lr 5.29e-04 | gnorm 0.16 | 7,087 tok/s | peak mem 11.0 GB
  EVAL step 125 | train 8.038 | val 8.050 | val ppl 3134.0
  saved checkpoint (best) at step 125 -> /kaggle/working/ckpt.pt
step   130 | loss 7.995 | lr 5.21e-04 | gnorm 0.15 | 7,083 tok/s | peak mem 11.0 GB
step   135 | loss 8.033 | lr 5.12e-04 | gnorm 0.15 | 7,085 tok/s | peak mem 11.0 GB
step   140 | loss 7.706 | lr 5.04e-04 | gnorm 0.15 | 7,082 tok/s | peak 